# Final Project DSCI-552


reference: 
1) https://matplotlib.org/stable/index.html
2) https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html
3) https://github.com/INTERMT/Awesome-Keras-Chinese

## Text Classification

(a) In this problem, we are trying to build a classifier to analyze the sentiment of
reviews. You are provided with text data in two folders: one folder involves
positive reviews, and one folder involves negative reviews

In [ ]:
import re
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import matplotlib.pyplot as plt

def read_txt_files(archive_path, label):
    rows = []
    with ZipFile(archive_path) as archive:
        for name in archive.namelist():
            match = re.search(r'cv(\d+)_', Path(name).name)
            if name.endswith('.txt') and match:
                rows.append({
                    'cv': int(match.group(1)),
                    'text': archive.read(name).decode('utf-8'),
                    'label': label,
                })
    return pd.DataFrame(rows)

def clean_text(text):
    if not isinstance(text, str):  
        return ''  # debug
    # remove non-alphabetical characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def split(df):
    # iii. The name of each text file starts with cv number. 
    # Use text files 0-699 in each class for training and 700-999 for testing
    df = df.copy()
    df['text'] = df['text'].apply(clean_text)
    training_df = df.loc[df['cv'] < 700].drop(columns='cv').reset_index(drop=True)
    testing_df = df.loc[df['cv'] >= 700].drop(columns='cv').reset_index(drop=True)
    return training_df, testing_df


### (b) Data Exploration and Pre-processing

In [ ]:
data_dir = Path.cwd() / 'Data'
if not data_dir.exists():
    data_dir = Path.cwd() / 'final_project' / 'Data'

neg_df = read_txt_files(data_dir / 'neg.zip', label=-1)
pos_df = read_txt_files(data_dir / 'pos.zip', label=1)

training_dfneg, testing_dfneg = split(neg_df)
training_dfpos, testing_dfpos = split(pos_df)

# Combine the two dataframe into 1
training = pd.concat([training_dfneg, training_dfpos], ignore_index=True)
training = training.sample(frac=1, random_state=42).reset_index(drop=True)
testing = pd.concat([testing_dfneg, testing_dfpos], ignore_index=True)

In [ ]:
training

In [ ]:
testing

In [ ]:
df = pd.concat([training,testing])
df

iv. Count the number of unique words in the whole dataset (train + test) and
print it out.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(binary=True) # set binary true to only count each word once
text = df['text'].to_list()
vectorizer.fit(text)
unique_words = vectorizer.get_feature_names_out()

# Print
print("the number of unique words in the whole dataset is", len(unique_words))

v. Calculate the average review length and the standard deviation of review
lengths. Report the results

In [ ]:
# Calculate average review length, split word, not count the string lenth
df['review_length'] = df['text'].apply(lambda x: len(x.split()))
average_length = df['review_length'].mean()
std_dev = df['review_length'].std()

print(f"The average review length is {average_length}, the standard deviation (SD) of review length is {std_dev}")

In [ ]:
df

vi. Plot the histogram of review lengths

In [ ]:
# set the number of bins with the *bins* keyword argument as 50, 
# the review lenth is obviously a skewed distribution
df['review_length'].hist(bins=50)

vii. To represent each text (= data point), there are many ways. In NLP/Deep
Learning terminology, this task is called tokenization. It is common to represent text using popularity/ rank of words in text. The most common word
in the text will be represented as 1, the second most common word will be
represented as 2, etc. Tokenize each text document using this method

reference: 

https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/text/Tokenizer

https://www.tensorflow.org/responsible_ai/fairness_indicators/tutorials/Fairness_Indicators_TFCO_Wiki_Case_Study


In [ ]:
import keras
print(keras.__version__)

from IPython.display import display
from IPython.display import HTML
import numpy as np
import pandas as pd

import tensorflow as tf
import tensorflow.keras as keras
tf.keras.utils.set_random_seed(42)
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.preprocessing import text
from tensorflow.keras.layers import Embedding, Flatten, Dense, Dropout, Conv1D, MaxPooling1D


In [ ]:
# tokenization
tokenizer = text.Tokenizer()
tokenizer.fit_on_texts(df["text"])
# check the result
tokenizer.word_index

viii. Select a review length L that 70% of the reviews have a length below it. If
you feel more adventurous, set the threshold to 90%.


ix. Truncate reviews longer than L words and zero-pad reviews shorter than L
so that all texts (= data points) are of length L.

In [ ]:
import numpy as np
# viii
review_lengths = training['text'].str.split().str.len().values
review_70_length = int(np.percentile(review_lengths, 70))
print("Value at the 70th percentile of review lengths:", review_70_length)

# ix
def truncate(texts, tokenizer, max_sequence_length):
    # Turns text into into padded sequences.
    text_sequences = tokenizer.texts_to_sequences(texts)
    
    # padding='post'
    # truncating='post'
    # padding is done with zeros at the end of sequences shorter than the specified maximum length 
    return sequence.pad_sequences(text_sequences, maxlen=max_sequence_length, padding='post', truncating='post')

# here max_sequence_length = percentile 70 (737)
text_df = truncate(df["text"], tokenizer, review_70_length)
training_x = truncate(training["text"], tokenizer, review_70_length)
testing_x = truncate(testing["text"], tokenizer, review_70_length)

In [ ]:
text_df.shape

# here we have 2000 document (text), and each document's length is 700 words

In [ ]:
training_x.shape

### (c) Word Embeddings

use a word embedding layer for this project. Assume that we are interested in the top 5,000 words. This means that in each integer sequence that
represents each document, we set to zero those integers that represent words
that are not among the top 5,000 words in the document.5 If you feel more
adventurous, use all the words that appear in this corpus. Choose the length
of the embedding vector for each word to be 32. Hence, each document is
represented as a 32 ×L matrix

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
def evaluation(ytrue, ypred):
    # for report result
    results = {
        'accuracy': [accuracy_score(ytrue, ypred)],
        'precision': [precision_score(ytrue, ypred, zero_division=0)],
        'recall': [recall_score(ytrue, ypred)],
        'f1': [f1_score(ytrue, ypred)],
    }
    
    return pd.DataFrame(results)

In [ ]:
def preprocess_documents(documents, top_words, embedding_dim):
    # Create a dictionary to map words to indices
    word_index = {}
    for doc in documents:
        for word in doc:
            word_index[word] = word_index.get(word, 0) + 1
            
    # Select the top 5000 words based on frequency
    top_words_list = sorted(word_index, key=word_index.get, reverse=True)[:top_words]
    # Create a word to index dictionary
    word_to_index = {word: idx + 1 for idx, word in enumerate(top_words_list)}
    # Convert documents to sequences of indices
    sequences = [[word_to_index.get(word, 0) for word in doc] for doc in documents]

    # Create the embedding matrix
    embedding_matrix = np.zeros((top_words + 1, embedding_dim))
    for word, idx in word_to_index.items():
        if idx > top_words:
            continue
        embedding_vector = np.random.rand(embedding_dim)  # initialize with random values or zeros
        embedding_matrix[idx] = embedding_vector

    # Create the embedding layer
    embedding_layer = Embedding(input_dim=top_words+1, output_dim=embedding_dim,
                                weights=[embedding_matrix], input_length=review_70_length)

    # Flatten the embedding layer
    flatten_layer = Flatten()

    return embedding_layer, flatten_layer

# Example usage
documents = text_df  # shape=(2000, 737)
top_words = 5000
embedding_dim = 32

embedding_layer, flatten_layer = preprocess_documents(documents, top_words, embedding_dim)

### (d) Multi-Layer Perceptron

Train a MLP with three (dense) hidden layers each of which has 50 ReLUs
and one output layer with a single sigmoid neuron. Use a dropout rate of
20% for the first layer and 50% for the other layers. Use ADAM optimizer
and binary cross entropy loss (which is equivalent to having a softmax in the
output). To avoid overfitting, just set the number of epochs as 2. Use a batch
size of 10

reference: https://keras-cn.readthedocs.io/en/latest/

[input] -> | hidden layer | -> 50 ReLUS -> | hidden layer | -> 50 ReLUS -> | hidden layer | -> 50 ReLUS -> |output layer (single sigmoid)|

In [ ]:
# define model parameters
max_words = 5000  # Specify the input dimension
epochs = 2
batch_size = 10

# tokenizer
tokenizer = text.Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(training["text"])

training_sequences = tokenizer.texts_to_sequences(training["text"])
testing_sequences = tokenizer.texts_to_sequences(testing["text"])

training_x = sequence.pad_sequences(training_sequences, maxlen=review_70_length, padding='post', truncating='post')
testing_x = sequence.pad_sequences(testing_sequences, maxlen=review_70_length, padding='post', truncating='post')

In [ ]:
import tensorflow as tf

''' 
def bipolar_sigmoid(x):
    return (2 / (1 + tf.exp(-x))) - 1

def custom_binary_crossentropy_with_labels(y_true, y_pred):
    # convert y_true from [-1, 1] to [0, 1]
    y_pred = (y_pred + 1) / 2
    y_true = (y_true + 1) / 2
    
    y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon())  # 避免log(0)
    loss = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
    return tf.reduce_mean(loss)
'''
# define model
def create_mlp():
    model = keras.models.Sequential()
    
    # add the embedding layer
    model.add(Embedding(max_words + 1, output_dim=32, input_length = review_70_length))
    model.add(Flatten())
    model.add(Dense(50, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(50, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(50, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))
    return model

# Create the MLP model
model = create_mlp()

# Compile the model
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

# model.summary()

# adjust the label setting since the binary_crossentropy is for 0 / 1 label 
# or use bipolar_sigmoid and custom binary cross_entropy
# Here I chose to adjust the label type(a more simple and easier way)
training['label'] = training['label'].apply(lambda x: 0 if x == -1 else 1)
testing['label'] = testing['label'].apply(lambda x: 0 if x == -1 else 1)

# Train the model
model.fit(training_x, training['label'].values, 
          epochs=epochs, 
          batch_size=batch_size)

predic = model.predict(training_x, batch_size=128)
predictions_binary = [int(p > 0.5) for p in predic.ravel()]
training_result = evaluation(training['label'].values, predictions_binary)
print(f'training set result: {training_result}')

predic = model.predict(testing_x, batch_size=128)
predictions_binary = [int(p > 0.5) for p in predic.ravel()]
test_result = evaluation(testing['label'].values, predictions_binary)
print(f'test set result: {test_result}')

### (e) One-Dimensional Convolutional Neural Network

i. After the embedding layer, insert a Conv1D layer. This convolutional layer
has 32 feature maps , and each of the 32 kernels has size 3, i.e. reads embedded
word representations 3 vector elements of the word embedding at a time. The
convolutional layer is followed by a 1D max pooling layer with a length and
stride of 2 that halves the size of the feature maps from the convolutional
layer. The rest of the network is the same as the neural network above.

ii. Report the train and test accuracies of this model

In [ ]:
def create_cnn():
    model = keras.models.Sequential()
    
    # Embedding layer
    model.add(Embedding(max_words + 1, output_dim=32, input_length=review_70_length))
    
    # Convolutional Layer
    model.add(Conv1D(32, 3, activation='relu', padding = 'same'))  # 32 feature maps and a kernel size of 3
    # 1D max pooling
    model.add(MaxPooling1D(2, padding = 'same'))  # Pool size and strides of 2

    # Flatten data
    model.add(Flatten())

    model.add(Dense(50, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(50, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(1, activation='sigmoid')) 
    return model

model = create_cnn()
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), 
              loss='binary_crossentropy', # or custom_binary_crossentropy_with_labels 
              metrics=['accuracy'])

'''
# Adjust the label setting since the binary_crossentropy is for 0 / 1 label
training['label'] = training['label'].apply(lambda x: 0 if x == -1 else 1)
testing['label'] = testing['label'].apply(lambda x: 0 if x == -1 else 1)
'''

# training and record the history
history = model.fit(training_x, training['label'].values, 
                    epochs=epochs, 
                    batch_size=batch_size,
                    validation_split=0.2)

# evaluation
train_acc1 = history.history['accuracy'][-2]
print("the training accuracy[epoch1]: {:.2f}%".format(train_acc1 * 100))

train_acc = history.history['accuracy'][-1]
print("the training accuracy[epoch2]: {:.2f}%".format(train_acc * 100))
test_loss, test_acc = model.evaluate(testing_x, testing['label'].values, verbose=0)
print("the held-out testing accuracy: {:.2f}%".format(test_acc * 100))

### (f) Long Short-Term Memory Recurrent Neural Network

Each word is represented to LSTM as a vector of 32 elements and the LSTM
is followed by a dense layer of 256 ReLUs. Use a dropout rate of 0.2 for both LSTM and the dense layer. Train the model using 10-50 epochs and batch
size of 10.

Report the train and test accuracies of this model.

In [ ]:
from tensorflow.keras.layers import LSTM

def create_lstm():
    model = keras.models.Sequential()
    # embedding layer
    model.add(Embedding(max_words + 1, output_dim=32, input_length=review_70_length))
    model.add(LSTM(32, dropout=0.2, recurrent_dropout=0.2))  # 32 LSTM units, add dropout to LSTM
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.2))  
    model.add(Dense(1, activation='sigmoid'))
    return model

model = create_lstm()
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

training['label'] = training['label'].apply(lambda x: 0 if x == -1 else 1)
testing['label'] = testing['label'].apply(lambda x: 0 if x == -1 else 1)

# train the model
history = model.fit(training_x, training['label'].values, 
                    epochs=10, 
                    batch_size=10,
                    validation_split=0.2)

# evaluation
train_acc = history.history['accuracy'][-1]
print("the training accuracy: {:.2f}%".format(train_acc * 100))
test_loss, test_acc = model.evaluate(testing_x, testing['label'].values, verbose=0)
print("the testing accuracy: {:.2f}%".format(test_acc * 100))
